# Phase 1 — Data Acquisition & Profiling

Profiles every raw dataset under `data/raw/` and produces the Phase 1 data-quality report table.

**Prerequisite:** populate `data/raw/<dataset>/` first, e.g.
```bash
python -m etl.extract.download_raw --all
python -m etl.extract.download_raw --dataset uae_trade --url <uae-open-data-url>
python -m etl.extract.download_raw --dataset uae_cpi --url <fcsc-cpi-url>
```
This notebook does not download data itself — see `etl/extract/download_raw.py` and `docs/business_requirements.md` §5 for the PUBLIC/SYNTHETIC/DERIVED labeling convention that every dataset here must carry.

All heavy logic lives in `src/data_quality/` (profiler, checks, report) — this notebook calls into it rather than reimplementing it, so `etl/validate/` can reuse the exact same rules in Phase 2.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.data_quality.checks import run_all_checks
from src.data_quality.dataset_specs import DATASET_SPECS
from src.data_quality.profiler import profile_dataframe, profile_to_frame
from src.data_quality.report import build_quality_report, save_issues, save_report

RAW_DIR = Path("../data/raw")
REPORT_DIR = Path("../reports/data_quality")

## 1. Load raw files

Each dataset key below maps to a CSV expected under `data/raw/`. Olist ships as several files (orders, order_items, products, ...); DataCo and the UAE sources ship as one file each. Missing files are skipped with a warning rather than raising, so this notebook stays runnable before every dataset is downloaded.

In [2]:
# dataset key -> path, relative to RAW_DIR
SOURCE_FILES = {
    "olist_orders": "olist/olist_orders_dataset.csv",
    "olist_order_items": "olist/olist_order_items_dataset.csv",
    "olist_products": "olist/olist_products_dataset.csv",
    "dataco_shipments": "dataco/DataCoSupplyChainDataset.csv",
    "uae_trade": "uae_trade/uae_trade.csv",
    "uae_cpi": "uae_cpi/uae_cpi.csv",
}

dataframes: dict[str, pd.DataFrame] = {}
for key, rel_path in SOURCE_FILES.items():
    path = RAW_DIR / rel_path
    if not path.exists():
        print(f"[skip] {key}: {path} not found — run etl.extract.download_raw first")
        continue
    # DataCo's public CSV is latin-1 encoded, not utf-8
    encoding = "latin-1" if key == "dataco_shipments" else "utf-8"
    dataframes[key] = pd.read_csv(path, encoding=encoding, low_memory=False)
    print(f"[loaded] {key}: {dataframes[key].shape}")

[loaded] olist_orders: (99441, 8)


[loaded] olist_order_items: (112650, 7)
[loaded] olist_products: (32951, 9)


[loaded] dataco_shipments: (180519, 53)
[loaded] uae_trade: (21593, 16)
[skip] uae_cpi: ..\data\raw\uae_cpi\uae_cpi.csv not found — run etl.extract.download_raw first


## 2. Verify column names against `dataset_specs.py`

Kaggle dataset revisions occasionally rename/re-case columns. Run this before trusting the checks below — if a `required_fields`/`id_columns`/`date_order_pairs` entry doesn't appear here, fix it in `src/data_quality/dataset_specs.py`, not in this notebook.

In [3]:
for key, df in dataframes.items():
    print(f"\n{key} columns:")
    print(list(df.columns))


olist_orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_order_items columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_products columns:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

dataco_shipments columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department I

## 3. Profile every loaded dataset

Row/column counts, dtypes, missingness, duplicates, unique values, min/max/mean/median/std.

In [4]:
profiles = {}
for key, df in dataframes.items():
    profiles[key] = profile_dataframe(df, key)
    print(f"\n=== {key} ===")
    print(f"rows={profiles[key].n_rows} cols={profiles[key].n_cols} "
          f"duplicate_rows={profiles[key].n_duplicate_rows} "
          f"overall_missing_pct={profiles[key].overall_missing_pct:.2f}%")
    display(profile_to_frame(profiles[key]))


=== olist_orders ===
rows=99441 cols=8 duplicate_rows=0 overall_missing_pct=0.62%


,column,dtype,missing_count,missing_pct,n_unique,min,max,mean,median,std
0,order_id,object,0,0.00,99441,None,None,None,None,None
1,customer_id,object,0,0.00,99441,None,None,None,None,None
2,order_status,object,0,0.00,8,None,None,None,None,None
3,order_purchase_timestamp,object,0,0.00,98875,None,None,None,None,None
4,order_approved_at,object,160,0.16,90733,None,None,None,None,None
5,order_delivered_carrier_date,object,1783,1.79,81018,None,None,None,None,None
6,order_delivered_customer_date,object,2965,2.98,95664,None,None,None,None,None
7,order_estimated_delivery_date,object,0,0.00,459,None,None,None,None,None



=== olist_order_items ===
rows=112650 cols=7 duplicate_rows=0 overall_missing_pct=0.00%


,column,dtype,missing_count,missing_pct,n_unique,min,max,mean,median,std
0,order_id,object,0,0.0,98666,NaN,NaN,NaN,NaN,NaN
1,order_item_id,int64,0,0.0,21,1.00,21.00,1.197834,1.00,0.705124
2,product_id,object,0,0.0,32951,NaN,NaN,NaN,NaN,NaN
3,seller_id,object,0,0.0,3095,NaN,NaN,NaN,NaN,NaN
4,shipping_limit_date,object,0,0.0,93318,NaN,NaN,NaN,NaN,NaN
5,price,float64,0,0.0,5968,0.85,6735.00,120.653739,74.99,183.633928
6,freight_value,float64,0,0.0,6999,0.00,409.68,19.990320,16.26,15.806405



=== olist_products ===
rows=32951 cols=9 duplicate_rows=0 overall_missing_pct=0.83%


,column,dtype,missing_count,missing_pct,n_unique,min,max,mean,median,std
0,product_id,object,0,0.00,32951,NaN,NaN,NaN,NaN,NaN
1,product_category_name,object,610,1.85,73,NaN,NaN,NaN,NaN,NaN
2,product_name_lenght,float64,610,1.85,66,5.0,76.0,48.476949,51.0,10.245741
3,product_description_lenght,float64,610,1.85,2960,4.0,3992.0,771.495285,595.0,635.115225
4,product_photos_qty,float64,610,1.85,19,1.0,20.0,2.188986,1.0,1.736766
5,product_weight_g,float64,2,0.01,2204,0.0,40425.0,2276.472488,700.0,4282.038731
6,product_length_cm,float64,2,0.01,99,7.0,105.0,30.815078,25.0,16.914458
7,product_height_cm,float64,2,0.01,102,2.0,105.0,16.937661,13.0,13.637554
8,product_width_cm,float64,2,0.01,95,6.0,118.0,23.196728,20.0,12.079047



=== dataco_shipments ===
rows=180519 cols=53 duplicate_rows=0 overall_missing_pct=3.51%


,column,dtype,missing_count,missing_pct,n_unique,min,max,mean,median,std
0,Type,object,0,0.00,4,NaN,NaN,NaN,NaN,NaN
1,Days for shipping (real),int64,0,0.00,7,0.000000,6.000000,3.497654,3.000000,1.623722
2,Days for shipment (scheduled),int64,0,0.00,4,0.000000,4.000000,2.931847,4.000000,1.374449
3,Benefit per order,float64,0,0.00,21998,-4274.979980,911.799988,21.974989,31.520000,104.433526
4,Sales per customer,float64,0,0.00,2927,7.490000,1939.989990,183.107609,163.990005,120.043670
5,Delivery Status,object,0,0.00,4,NaN,NaN,NaN,NaN,NaN
6,Late_delivery_risk,int64,0,0.00,2,0.000000,1.000000,0.548291,1.000000,0.497664
7,Category Id,int64,0,0.00,51,2.000000,76.000000,31.851451,29.000000,15.640064
8,Category Name,object,0,0.00,50,NaN,NaN,NaN,NaN,NaN
9,Customer City,object,0,0.00,563,NaN,NaN,NaN,NaN,NaN



=== uae_trade ===
rows=21593 cols=16 duplicate_rows=0 overall_missing_pct=19.57%


,column,dtype,missing_count,missing_pct,n_unique,min,max,mean,median,std
0,DATAFLOW,object,0,0.00,1,NaN,NaN,NaN,NaN,NaN
1,REF_AREA,object,0,0.00,1,NaN,NaN,NaN,NaN,NaN
2,FREQ,object,0,0.00,1,NaN,NaN,NaN,NaN,NaN
3,UNIT_MEASURE,object,0,0.00,1,NaN,NaN,NaN,NaN,NaN
4,SOURCE_DETAIL,object,0,0.00,1,NaN,NaN,NaN,NaN,NaN
5,MEASURE,object,0,0.00,1,NaN,NaN,NaN,NaN,NaN
6,TRADE_TYPE,object,0,0.00,1,NaN,NaN,NaN,NaN,NaN
7,HS_SECTION,object,0,0.00,1,NaN,NaN,NaN,NaN,NaN
8,HS_CHAPTER,object,0,0.00,1,NaN,NaN,NaN,NaN,NaN
9,COUNTRY,object,0,0.00,254,NaN,NaN,NaN,NaN,NaN


## 4. Run data-quality checks

Nulls in required fields, negative values, impossible dates (delivery before order), duplicate IDs, and IQR/Z-score outliers — driven by `DATASET_SPECS`.

In [5]:
issues_by_dataset = {}
for key, df in dataframes.items():
    spec = DATASET_SPECS.get(key)
    if spec is None:
        print(f"[skip] no DatasetSpec for {key}")
        continue
    issues = run_all_checks(df, spec)
    issues_by_dataset[key] = issues
    print(f"\n=== {key}: {len(issues)} issue type(s) ===")
    for issue in issues:
        print(f"  [{issue.severity:8}] {issue.check:16} {issue.column}: {issue.detail}")


=== olist_orders: 0 issue type(s) ===

=== olist_order_items: 4 issue type(s) ===
  [warning ] outlier_iqr      price: 8427 values outside [-102.60, 277.40]
  [warning ] outlier_iqr      freight_value: 12134 values outside [0.98, 33.25]
  [warning ] outlier_zscore   price: 1966 values with |z| > 3.0
  [warning ] outlier_zscore   freight_value: 2041 values with |z| > 3.0

=== olist_products: 0 issue type(s) ===



=== dataco_shipments: 6 issue type(s) ===
  [warning ] outlier_iqr      Sales: 488 values outside [-149.98, 569.91]
  [warning ] outlier_iqr      Order Item Product Price: 2048 values outside [-174.99, 424.98]
  [warning ] outlier_iqr      Benefit per order: 18942 values outside [-79.70, 151.50]
  [warning ] outlier_zscore   Sales: 467 values with |z| > 3.0
  [warning ] outlier_zscore   Order Item Product Price: 488 values with |z| > 3.0
  [warning ] outlier_zscore   Benefit per order: 3608 values with |z| > 3.0

=== uae_trade: 2 issue type(s) ===
  [warning ] outlier_iqr      OBS_VALUE: 3554 values outside [-67248498.23, 112375395.72]
  [warning ] outlier_zscore   OBS_VALUE: 362 values with |z| > 3.0


## 5. Build and save the Phase 1 data-quality report

In [6]:
report_df = build_quality_report(list(profiles.values()), issues_by_dataset)
display(report_df)

csv_path, md_path = save_report(report_df, REPORT_DIR)
issues_path = save_issues(issues_by_dataset, REPORT_DIR)
print(f"Saved: {csv_path}, {md_path}, {issues_path}")

,dataset,rows,columns,missing_pct,duplicate_rows,critical_issues,warning_issues,verdict
0,olist_orders,99441,8,0.62,0,0,0,PASS
1,olist_order_items,112650,7,0.00,0,0,24568,WARN
2,olist_products,32951,9,0.83,0,0,0,PASS
3,dataco_shipments,180519,53,3.51,0,0,26041,WARN
4,uae_trade,21593,16,19.57,0,0,3916,WARN


Saved: ..\reports\data_quality\data_quality_report.csv, ..\reports\data_quality\data_quality_report.md, ..\reports\data_quality\data_quality_issues.csv


## Next steps

Any `FAIL` verdict above must be resolved (or explicitly accepted with a documented reason) before Phase 2 loads this data into `staging`. See `docs/implementation_plan.md` Week 3 for the ETL build order.